In [1]:
'''The first thing we'll do in preprocessing is decide which columns should actually enter our ML model, especially:

nameOrig
nameDest
newbalanceOrig
newbalanceDest
isFlaggedFraud

This is an important step because we want a model that is realistic and doesn't suffer from data leakage.'''

"The first thing we'll do in preprocessing is decide which columns should actually enter our ML model, especially:\n\nnameOrig\nnameDest\nnewbalanceOrig\nnewbalanceDest\nisFlaggedFraud\n\nThis is an important step because we want a model that is realistic and doesn't suffer from data leakage."

In [2]:
import pandas as pd

file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(file_path)

print(df.shape)
print(df.columns.tolist())

(6362620, 11)
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [4]:
print(df.columns.tolist())
'''then don't delete anything yet.

We're going to decide the ML features carefully because:

newbalanceOrig / newbalanceDest can introduce post-transaction leakage
nameOrig / nameDest have extremely high cardinality
isFlaggedFraud is an existing rule-based flag
isFraud is our target'''

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


"then don't delete anything yet.\n\nWe're going to decide the ML features carefully because:\n\nnewbalanceOrig / newbalanceDest can introduce post-transaction leakage\nnameOrig / nameDest have extremely high cardinality\nisFlaggedFraud is an existing rule-based flag\nisFraud is our target"

In [5]:
''' firstly remove the target column it directly leaks data'''
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6362620, 10)
y shape: (6362620,)


In [10]:
'''Remove the columns we don't want the model to use

For our first realistic model, we'll exclude:

nameOrig → millions of unique account IDs
nameDest → millions of unique receiver IDs
newbalanceOrig → balance after transaction
newbalanceDest → balance after transaction'''
#isFlaggedFraud → an existing rule-based fraud flag; we'll keep it for analysis, but not use it as a main ML feature''''

features_to_drop = [
    "nameOrig",
    "nameDest",
    "newbalanceOrig",
    "newbalanceDest",
    "isFlaggedFraud"
]

X = X.drop(columns=features_to_drop)

print("Selected features:")
print(X.columns.tolist())

print("\nX shape:", X.shape)

Selected features:
['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']

X shape: (6362620, 5)


In [11]:
'''Now the next issue is the type column.

Currently:

step              → number
type              → text ❌
amount            → number
oldbalanceOrg     → number
oldbalanceDest    → number

Most ML algorithms need numerical input, so we need to encode type.'''
print(X["type"].value_counts())


type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [12]:
'''Since type is categorical (CASH_OUT, PAYMENT, etc.), we'll use one-hot encoding.

This converts:

type
CASH_OUT
PAYMENT
TRANSFER

into numerical columns such as:

type_CASH_OUT   type_PAYMENT   type_TRANSFER
1               0              0
0               1              0
0               0              1

For this project, use pandas' get_dummies().'''
X = pd.get_dummies(X, columns=["type"], dtype=int)

print("Columns after encoding:")
print(X.columns.tolist())

print("\nX shape:", X.shape)

Columns after encoding:
['step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']

X shape: (6362620, 9)


In [13]:
print(X.dtypes)
'''We want to confirm that all 9 features are numerical.

You should see something similar to:

step               int64
amount           float64
oldbalanceOrg    float64
oldbalanceDest   float64
type_CASH_IN       int64
type_CASH_OUT     int64
type_DEBIT         int64
type_PAYMENT       int64
type_TRANSFER     int64

Why this matters:

Before:
type → "CASH_OUT" ❌ text

After:
type_CASH_OUT → 0 or 1 ✅'''

step                int64
amount            float64
oldbalanceOrg     float64
oldbalanceDest    float64
type_CASH_IN        int64
type_CASH_OUT       int64
type_DEBIT          int64
type_PAYMENT        int64
type_TRANSFER       int64
dtype: object


'We want to confirm that all 9 features are numerical.\n\nYou should see something similar to:\n\nstep               int64\namount           float64\noldbalanceOrg    float64\noldbalanceDest   float64\ntype_CASH_IN       int64\ntype_CASH_OUT     int64\ntype_DEBIT         int64\ntype_PAYMENT       int64\ntype_TRANSFER     int64\n\nWhy this matters:\n\nBefore:\ntype → "CASH_OUT" ❌ text\n\nAfter:\ntype_CASH_OUT → 0 or 1 ✅'

In [14]:
#check for missing values 
print("Missing values in X:")
print(X.isnull().sum().sum())

Missing values in X:
0


In [15]:
'''Split the data into training and testing sets

We need to keep some data aside to test the model on transactions it hasn't seen during training.

Because fraud is extremely rare, we'll use stratified splitting so both sets maintain roughly the same fraud percentage.'''
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

ModuleNotFoundError: No module named 'sklearn'

In [16]:
import sklearn

print(sklearn.__version__)

1.9.1


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5090096, 9)
X_test : (1272524, 9)
y_train: (5090096,)
y_test : (1272524,)


In [18]:
print("Training fraud distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting fraud distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

Training fraud distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64

Testing fraud distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64
isFraud
0    99.870887
1     0.129113
Name: proportion, dtype: float64


In [19]:
normal_count = (y_train == 0).sum()
fraud_count = (y_train == 1).sum()

print("Normal transactions:", normal_count)
print("Fraud transactions:", fraud_count)
print("Imbalance ratio:", normal_count / fraud_count)

Normal transactions: 5083526
Fraud transactions: 6570
Imbalance ratio: 773.7482496194825


In [21]:
'''Exactly. 👍 Your calculation confirms the extreme class imbalance:

Normal: 5,083,526
Fraud: 6,570
Ratio: ~774 normal transactions for every 1 fraud transaction

This is why accuracy alone will be misleading. A model could predict almost everything as normal and still appear highly accurate.

Step 12 — We'll use class weighting

For the first model, we won't create millions of synthetic/duplicate fraud rows. Instead, we'll use class_weight="balanced" with models that support it.

Conceptually:

Normal transaction → lower weight
Fraud transaction  → higher weight

This tells the model:

"Making a mistake on a fraud transaction should matter much more."

Before training, let's verify the class weights that scikit-learn would assign.'''
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

print("Normal class weight:", weights[0])
print("Fraud class weight :", weights[1])

Normal class weight: 0.5006462050159672
Fraud class weight : 387.37412480974126


In [22]:
import pandas as pd

file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (6362620, 11)
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [23]:
# Separate features and target

X = df.drop(columns=["isFraud"])
y = df["isFraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6362620, 10)
y shape: (6362620,)
